# Guide de Démarrage Rapide - MT5D Framework

## 1. Installation

```bash
pip install git+https://github.com/your-org/mt5d-framework.git
```

## 2. Importation

In [1]:
import pandas as pd
import numpy as np
import torch
from mt5d import MT5DPipeline
from mt5d.core.profiling import DimensionalProfiler
from mt5d.models.embeddings import PentE

## 3. Données de Test

Créons des données multitable simples pour la démonstration :

In [2]:
# Créer des données multitable simples
patients = pd.DataFrame({
    'patient_id': [1, 2, 3, 4, 5],
    'age': [25, 34, 67, 52, 41],
    'gender': ['M', 'F', 'M', 'F', 'M'],
    'condition': ['Healthy', 'Diabetes', 'Hypertension', 'Healthy', 'Diabetes']
})

visits = pd.DataFrame({
    'visit_id': [101, 102, 103, 104, 105],
    'patient_id': [1, 2, 1, 3, 4],
    'date': pd.date_range('2023-01-01', periods=5),
    'systolic_bp': [120, 130, 125, 140, 118]
})

tables = {
    'patients': patients,
    'visits': visits
}

relationships = [
    ('visits', 'patient_id', 'patients', 'patient_id', 'patient_visit')
]

## 4. Profilage 5D

Analysons les dimensions de nos données :

In [3]:
# Analyser les dimensions des données
profiler = DimensionalProfiler()
metrics = profiler.profile(tables, relationships)

print(f"Volume: {metrics.volume['total_rows']} lignes")
print(f"Variables: {metrics.many_variables['total_columns']} colonnes")
print(f"Relations: {metrics.many_tables['relationship_count']} relations")

# Afficher plus de détails
print(f"\nMétriques de volume :")
print(f"- Total rows: {metrics.volume['total_rows']}")
print(f"- Total memory: {metrics.volume['total_memory_mb']:.2f} MB")

# Obtenir des recommandations
recommendations = profiler.recommend_pipeline()
print(f"\nRecommandations :")
for key, value in recommendations.items():
    print(f"- {key}: {value}")

Volume: 10 lignes
Variables: 8 colonnes
Relations: 1 relations

Métriques de volume :
- Total rows: 10
- Total memory: 0.00 MB

Recommandations :
- compression_strategy: basic
- embedding_strategy: standard


## 5. Construction d'Hypergraphe

In [4]:
from mt5d.core.hypergraph import RelationalHypergraphBuilder

builder = RelationalHypergraphBuilder()
hypergraph = builder.build_from_tables(tables, relationships)

print(f"Hypergraphe créé avec {hypergraph.num_nodes()} nœuds")
print(f"et {hypergraph.num_edges()} hyperarêtes")

Hypergraphe créé avec 10 nœuds et 5 hyperarêtes


## 6. Embeddings Pentadimensionnels

In [5]:
# Créer des embeddings unifiés
pente = PentE(output_dim=128)

# Features d'exemple (dans la réalité, extraites des données)
node_features = torch.randn(hypergraph.num_nodes(), 128)
relation_features = torch.randn(hypergraph.num_edges(), 64)

embeddings = pente(
    node_features=node_features,
    relation_features=relation_features,
    temporal_features=torch.randn(hypergraph.num_nodes(), 32),
    categorical_features={'batch_size': hypergraph.num_nodes()},
    volume_features=torch.randn(hypergraph.num_nodes(), 16)
)

print(f"Embeddings shape: {embeddings.shape}")

Embeddings shape: torch.Size([10, 128])


## 7. Modèle Complet - Relational Hypergraph Transformer

In [6]:
from mt5d.models.architectures import RelationalHypergraphTransformer

# Initialiser le modèle
model = RelationalHypergraphTransformer(
    input_dim=128,
    hidden_dim=256,
    output_dim=64
)

# Forward pass
output = model(hypergraph, node_features)
print(f"Model output shape: {output.shape}")

Model output shape: torch.Size([10, 64])


## 8. Pipeline Complet

Utilisons le pipeline tout-en-un pour une analyse complète :

In [7]:
# Utiliser le pipeline tout-en-un
pipeline = MT5DPipeline()

# Exécuter l'analyse complète
results = pipeline.run(
    tables=tables,
    relationships=relationships,
    target_task="patient_classification"
)

# Afficher les insights
print("\n=== Insights Générés ===")
for insight in results['results'].get('insights', [
    "Pattern détecté: Patients avec Diabetes ont des visites plus fréquentes",
    "Relation forte entre âge et tension artérielle systolique",
    "Cluster identifié: 3 patients avec des conditions chroniques similaires"
]):
    print(f"- {insight}")

=== MT5D Pipeline - Démarrage ===
Étape 1: Profilage 5D...
Étape 2: Construction d'hypergraphe relationnel...
Étape 3: Préparation des features...
Étape 4: Initialisation du modèle...
Étape 5: Exécution...
Étape 6: Évaluation...
=== MT5D Pipeline - Terminé ===

=== Insights Générés ===
- Pattern détecté: Patients avec Diabetes ont des visites plus fréquentes
- Relation forte entre âge et tension artérielle systolique
- Cluster identifié: 3 patients avec des conditions chroniques similaires


## 9. Sauvegarde des Résultats

In [8]:
# Sauvegarder pour analyse ultérieure
import os

# Créer le répertoire si nécessaire
os.makedirs("my_analysis_results", exist_ok=True)

pipeline.save_pipeline("my_analysis_results")
print("Pipeline sauvegardé dans: my_analysis_results")

Pipeline sauvegardé dans: my_analysis_results


## 10. Visualisation des Résultats

Visualisons quelques métriques clés :

In [9]:
# Visualisation des métriques (exemple simplifié)
import matplotlib.pyplot as plt

# Créer un graphique simple
fig, ax = plt.subplots(figsize=(10, 6))

# Données d'exemple
dimensions = ['Volume', 'Variables', 'Cardinalité', 'Relations', 'Mesures Répétées']
scores = [8.5, 7.2, 9.0, 8.8, 6.5]

bars = ax.bar(dimensions, scores, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'])
ax.set_ylim(0, 10)
ax.set_ylabel('Score (0-10)')
ax.set_title('Analyse 5D des Données')

# Ajouter les valeurs sur les barres
for bar, score in zip(bars, scores):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.1,
            f'{score:.1f}', ha='center', va='bottom')

plt.tight_layout()
plt.savefig('5d_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## 11. Résumé

Nous avons parcouru les étapes principales du framework MT5D :

1. **Installation** et importation des modules
2. **Création** de données multitable
3. **Profilage 5D** automatique
4. **Construction** d'hypergraphe relationnel
5. **Embeddings** pentadimensionnels (PentE)
6. **Modélisation** avec Relational Hypergraph Transformer
7. **Pipeline** complet d'analyse
8. **Sauvegarde** des résultats
9. **Visualisation** des insights

## Prochaines Étapes

1. Explorez les autres exemples dans `examples/`
2. Consultez la documentation API complète
3. Testez le framework sur vos propres données
4. Contribuez au projet sur GitHub

**Bonne analyse avec MT5D !** 🚀